# testLeague — loading a season through the ESPN adapter

Loads a season with `EspnLeagueRepository` and looks at the resulting domain objects
(`League`, `Team`, `Matchup`, `Lineup`), plus a few sanity checks against ESPN's own numbers.

Tables come from `fantasy_analysis.application.frames`: `teams_frame`, `matchups_frame`,
`team_results_frame`, `standings_frame`, `schedule_frame`, `lineup_frame`,
`lineup_entries_frame`, `bench_points_frame`.

In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
from collections import Counter

import pandas as pd

from fantasy_analysis.application.frames import (
    bench_points_frame,
    lineup_entries_frame,
    lineup_frame,
    matchups_frame,
    schedule_frame,
    standings_frame,
    team_results_frame,
    teams_frame,
)
from fantasy_analysis.config import settings
from fantasy_analysis.infrastructure.espn import EspnLeagueRepository

pd.set_option("display.max_columns", 50)

SEASON = 2026

## Load the season

Makes several ESPN requests per week, so a full season takes a little while.

In [35]:
repo = EspnLeagueRepository.from_settings(settings)
league = repo.get_league(SEASON)

weeks = sorted({m.week for m in league.matchups})
print(f"{league.name} ({league.season}): {league.num_teams} teams, weeks {weeks[0]}-{weeks[-1]}")
print("matchups by kind:", dict(Counter(m.kind for m in league.matchups)))

The Corrupt League (2026): 10 teams, weeks 1-7
matchups by kind: {'REGULAR_SEASON': 35}


In [47]:
league

League(league_id=1416732276, name='The Corrupt League', season=2026, teams=[Team(team_id=1, name='Chase You to the Playoffs', owners=['Sam Levitt']), Team(team_id=2, name='On a Train Back to Boston', owners=['Isaac Leibowitz']), Team(team_id=3, name='The Williams Family', owners=["Ryan O'Connor"]), Team(team_id=4, name='Will Power', owners=['ben englander']), Team(team_id=5, name='I didnt hear no bell', owners=['Gavriel Warren']), Team(team_id=6, name='#EATITFROMTHEBACK', owners=['Samuel Stein']), Team(team_id=7, name='Chubland', owners=['Moshe Gesin']), Team(team_id=8, name='Benevolent Commish', owners=['Aaron Fandel']), Team(team_id=9, name='The Germiphobes', owners=['Richard Kinches']), Team(team_id=10, name='To Infinity and Bijan', owners=['Jake Birnbach'])])

## Teams

In [48]:
teams_frame(league)

,team_id,name,owners,weeks_with_lineup
0,1,Chase You to the Playoffs,Sam Levitt,7
1,2,On a Train Back to Boston,Isaac Leibowitz,7
2,3,The Williams Family,Ryan O'Connor,7
3,4,Will Power,ben englander,7
4,5,I didnt hear no bell,Gavriel Warren,7
5,6,#EATITFROMTHEBACK,Samuel Stein,7
6,7,Chubland,Moshe Gesin,7
7,8,Benevolent Commish,Aaron Fandel,7
8,9,The Germiphobes,Richard Kinches,7
9,10,To Infinity and Bijan,Jake Birnbach,7


## Matchups

In [49]:
matchups_frame(league)

,week,kind,home,home_score,away,away_score,winner
0,1,REGULAR_SEASON,Chase You to the Playoffs,110.56,#EATITFROMTHEBACK,136.56,#EATITFROMTHEBACK
1,1,REGULAR_SEASON,Will Power,136.16,Benevolent Commish,166.40,Benevolent Commish
2,1,REGULAR_SEASON,The Germiphobes,125.26,On a Train Back to Boston,128.92,On a Train Back to Boston
3,1,REGULAR_SEASON,I didnt hear no bell,95.52,To Infinity and Bijan,120.06,To Infinity and Bijan
4,1,REGULAR_SEASON,Chubland,113.26,The Williams Family,159.76,The Williams Family
5,2,REGULAR_SEASON,The Williams Family,108.32,On a Train Back to Boston,136.86,On a Train Back to Boston
6,2,REGULAR_SEASON,Chase You to the Playoffs,128.32,Benevolent Commish,114.96,Chase You to the Playoffs
7,2,REGULAR_SEASON,To Infinity and Bijan,99.70,#EATITFROMTHEBACK,116.50,#EATITFROMTHEBACK
8,2,REGULAR_SEASON,Will Power,143.08,Chubland,127.64,Will Power
9,2,REGULAR_SEASON,I didnt hear no bell,86.42,The Germiphobes,96.88,The Germiphobes


## Standings

Regular season by default; pass `kind="PLAYOFF"` or `"CONSOLATION"` for the others.

In [38]:
standings_frame(league)

,wins,losses,ties,points_for,points_against
team,,,,,
#EATITFROMTHEBACK,6,1,0,950.86,748.00
The Williams Family,5,2,0,873.48,781.48
The Germiphobes,5,2,0,864.42,778.60
To Infinity and Bijan,5,2,0,790.02,760.42
Chubland,3,4,0,901.70,961.78
Benevolent Commish,3,4,0,895.24,857.80
On a Train Back to Boston,3,4,0,705.78,826.46
Chase You to the Playoffs,2,5,0,822.68,854.28
I didnt hear no bell,2,5,0,723.46,812.40


## Team results

One row per team per game: both sides of every matchup.

In [ ]:
team_results_frame(league)

## One team's week: lineup with actual vs. projected points

In [39]:
weeks

[1, 2, 3, 4, 5, 6, 7]

In [40]:
team = league.teams[9]
WEEK = 7
lineup = team.lineups[WEEK]

print(f"{team.name}, week {WEEK}: {lineup.points:.2f} points from starters")
lineup_frame(lineup)

To Infinity and Bijan, week 7: 125.42 points from starters


,slot,player,position,pro_team,points,projected,starter
0,RB,Bijan Robinson,RB,ATL,14.70,19.26,True
1,WR,George Pickens,WR,DAL,13.70,10.69,True
2,RB,Ashton Jeanty,RB,LV,17.60,15.16,True
3,BENCH,Lamar Jackson,QB,BAL,0.00,0.00,False
4,WR,Luther Burden III,WR,CHI,21.70,10.22,True
5,BENCH,Parker Washington,WR,JAX,0.00,0.00,False
6,BENCH,TreVeyon Henderson,RB,NE,3.30,6.62,False
7,TE,Tucker Kraft,TE,GB,12.80,9.64,True
8,WR,Jordan Addison,WR,MIN,3.30,9.06,True
9,FLEX,RJ Harvey,RB,DEN,13.00,9.50,True


In [41]:
# That team's schedule
schedule_frame(league, team)

,week,opponent,result
0,1,I didnt hear no bell,W
1,2,#EATITFROMTHEBACK,L
2,3,On a Train Back to Boston,W
3,4,Chase You to the Playoffs,W
4,5,Chubland,L
5,6,The Williams Family,W
6,7,Will Power,W


## Every lineup entry, season-long

One row per player per team per week — the main table for analysis.

In [42]:
entries_df = lineup_entries_frame(league)
print(entries_df.shape)
entries_df.head()

(1170, 10)


,week,team,slot,starter,player_id,player,position,pro_team,points,projected
0,1,Chase You to the Playoffs,WR,True,4362628,Ja'Marr Chase,WR,CIN,2.20,16.44
1,1,Chase You to the Playoffs,RB,True,3043078,Derrick Henry,RB,BAL,34.80,15.72
2,1,Chase You to the Playoffs,WR,True,4426502,Drake London,WR,ATL,4.50,11.04
3,1,Chase You to the Playoffs,QB,True,3918298,Josh Allen,QB,BUF,35.66,19.32
4,1,Chase You to the Playoffs,FLEX,True,4696981,Cam Skattebo,RB,NYG,14.10,12.37


In [43]:
# Points left on the bench, per team
bench_points_frame(league)

team
Chase You to the Playoffs    367.70
#EATITFROMTHEBACK            347.78
To Infinity and Bijan        303.48
Benevolent Commish           292.70
The Germiphobes              244.56
Chubland                     229.80
I didnt hear no bell         215.56
On a Train Back to Boston    211.90
Will Power                   210.26
The Williams Family          144.66
Name: points, dtype: float64

## Sanity checks

In [44]:
# ESPN's matchup score should equal the sum of starters' points
mismatches = [
    (m.week, side.name, score, round(side.lineups[m.week].points, 2))
    for m in league.matchups
    for side, score in ((m.home, m.home_score), (m.away, m.away_score))
    if side is not None and score is not None and abs(score - side.lineups[m.week].points) > 0.01
]
print("score mismatches:", mismatches or "none")

# Every team should have a lineup in every week it played
missing = [
    (m.week, side.name)
    for m in league.matchups
    for side in (m.home, m.away)
    if side is not None and m.week not in side.lineups
]
print("missing lineups:", missing or "none")

# Lineups with an empty starting slot (bench > 6)
short = entries_df[entries_df["starter"]].groupby(["week", "team"]).size()
print("lineups with an empty starting slot:")
short[short < 10]

score mismatches: none
missing lineups: none
lineups with an empty starting slot:


week  team              
4     Benevolent Commish    9
dtype: int64